In [27]:
import importlib
import polars as pl, polars.selectors as cs
from polars import col as c
import pyfixest as pf
from codigo.costa_rica import filters_cr 
from definitions import bases, results
from codigo import display_results as dis_res
from pathlib import Path
from ucr_tools.core import data_analysis as da
import os

In [6]:
#Definitions
tables_path = results.joinpath('cr/regresiones_dependientes')
general_output = tables_path.joinpath('regresion_dependientes.docx')
xl_employment = tables_path.joinpath('employment.xlsx')
xl_wages = tables_path.joinpath('wages.xlsx')

#### REGRESIONES PERSONAS DEPENDIENTES

In [ ]:
file = bases.joinpath(r'cr\limpio\cr_region_grouped_processed.sav')
df, var_labels, col_labels = da.read_file(file)
df:pl.DataFrame
dis_res.var_labels = var_labels

endogena_emp = 'Employment'
endogena_wage = 'Wages'

#--------------------------------------
# REGRESIONES PERSONAS DEPENDIENTES
#--------------------------------------
min_skill = 1
#Empleo
reg_df_emp = (
    df
    .filter(filters_cr.employment_filters(), filters_cr.filter_exogenas(min_skill=min_skill))
    .with_columns(
        c.condact.replace({2:0, 3:0}),
        )
    .rename({'condact':endogena_emp})
    ).clone()

#Salario
reg_df_wage = (
    df
    .filter(filters_cr.wage_filters(), filters_cr.filter_exogenas(min_skill=min_skill))
    .rename({'w_real_log':endogena_wage})
    ).clone()

#Diccionario de variable dependiente y dataframe
dfs = {
    endogena_emp:reg_df_emp,
    endogena_wage:reg_df_wage
}


fixed_effects = 'region + anno'
cluster = 'region'
results={}
dependents = {
    0:'No_Dependents',
    1:'Dependents'
}

print(f'DataBase -> {file.name}')
for var, _df in dfs.items():
    for i in [1,0]:
        print(f'\nRunning regressions\nVariable -> {var}\nGroup -> {dependents.get(i)}')
        reg_df = _df.filter(c.id_dependientes.eq(i))
        results[f'{var}_{dependents.get(i)}'] = pf.feols(f'{var} ~ {filters_cr.exogenas} | {fixed_effects}', data=reg_df, vcov={'CRV1': cluster})
        print(f'\nDone!\n{"*"*20}')

dis_res.generate_observations_tables([reg_df_emp, reg_df_wage], [xl_employment, xl_wages])
dis_res.great_table_to_word(pf.etable(results.values(), model_heads=['Dependents', 'Non Dependents', 'Dependents', 'Non Dependents'], **dis_res.table_args), output_path=general_output)

print(f'Files are avialable in {tables_path}')
print(f'Fixed Effects -> {fixed_effects}')
print(f'Exogenas -> {filters_cr.exogenas}')

#### REGRESION PROBABILIDAD DE TRABAJAR PARA UN HOGAR - COSTARRICENSES VS NICARAGUENSES

In [ ]:
#Crear identificador de ocupacion
df_2023 = (
    df.filter(c.anno.eq(2023))
    .with_columns(
        pl.when(c.ramafuerzatrab.eq(20)).then(1) #20 Hogar 99 no se puede clasificar (se incluye con el resto)
        .when(c.ramafuerzatrab.is_null()).then(None) #NA se mantiene nulo
        .otherwise(0)
        .alias('id_trabajo_hogar'),
    )
    .filter(
        ~c.a5.is_in([98,99]), #Filtrar valores no reportados
        c.nivinst.ne(99),
        c.a6.ne(9),
        c.a4.eq(2), # Mantener mujeres
        c.nacionalidad.is_in([0,1])
    )
)

exogenas = 'nacionalidad nivinst a5 a6 id_dependientes'.split()
exogenas_formula = f'nacionalidad + C(nivinst) + a5 + pow(a5, 2) + C(a6) + id_dependientes + C(region)'
fixed_effects = f'region'
cluster = 'region'

model = pf.feols(f'id_trabajo_hogar~{exogenas_formula}', data=df_2023, vcov='hetero')
table_args = {
    'type':'gt',
    'keep':'nacionalidad',
    'fe_present':'YES', 
    'thousands_sep':True,
    }

dis_res.great_table_to_word(pf.etable(model, **table_args), tables_path.joinpath('househould_employees.docx'))

#### REGRESION FORMALIDAD COSTA RICA

In [58]:
file = bases.joinpath(r'cr\limpio\cr_region_grouped_processed.sav')
df, var_labels, col_labels = da.read_file(file, output_format='polars')
df_2023 = df.filter(c.anno.eq(2023))

df_2023 = (
    df_2023
    .with_columns(
        pl.when(c.ramaemppri.is_in([20, 18, 19, 1, 9, 7, 3])).then('ramaemppri')
        .when(c.ramaemppri.is_null() | c.ramaemppri.eq(99)).then(None)
        .otherwise(0)
        .alias('rama_2')
    )
)

asegurados_list = [1, 8, 9, 12, 14]
df_2023 = df_2023.with_columns(
    pl.when(c.a11.is_in(asegurados_list))
    .then(1)
    .when(c.a11.eq(99) | c.a11.is_null())
    .then(None)
    .otherwise(0)
    .alias('formal_flag'),
    )

#Filters
filters = (
    c.nacionalidad.is_in([0,1]),
    c.nivinst.ne(99),
    ~c.a5.is_in([98,99]), #Filtrar valores no reportados
    c.a6.ne(9), #a6 estado conyugal
    c.a4.eq(2), # Mantener mujeres
    c.ramaemppri.ne(99),
    c.condact.eq(1),
)

df_2023 = df_2023.filter(filters)

#Hacerlo mas bien con una lista
exogenas = 'nacionalidad nivinst a5 a6 id_dependientes ramaemppri region'.split()
exogenas_formula = 'nacionalidad + C(nivinst) + a5 + pow(a5,2) + C(a6) + id_dependientes + C(region)'
endogena = 'formal_flag'

model = pf.feols(f'{endogena}~{exogenas_formula}', data=df_2023, vcov='hetero')
output = results.joinpath('cr/other_regressions/formalidad_cr.docx')
dis_res.great_table_to_word(pf.etable(model, **table_args), output)
os.startfile(output)

#### REGRESION FORMALIDAD CHILE

In [31]:
file = bases.joinpath(r'chile\limpio\chile_region_processed.sav')
df, var_labels, col_labels = da.read_file(file, output_format='polars')
df_2022 = df.filter(c.anno.eq(2022))

table_args = {
    'type':'gt',
    'keep':'nacionalidad',
    'fe_present':'YES', 
    'thousands_sep':True,
    }

#Filters
filters = (
    c.educ.is_between(0,12),
    c.nacionalidad.is_in([0,1]),
    c.edad.is_between(15,65, closed='both'),
    c.ecivil.is_between(1,8),
    c.rama1.is_between(1,21),
    c.activ.eq(1),
    c.sexo.eq(2),
    c.cotiza.is_in([0,1]),
)

df_2022 = df_2022.filter(filters)
df_2022 = df_2022.with_columns(c.cotiza.replace({0:1, 1:0}))

exogenas = 'nacionalidad educ edad ecivil id_dependientes rama1'
exogenas_formula = 'nacionalidad + C(educ) + edad + pow(edad, 2) + C(ecivil) + C(rama1) + id_dependientes + C(region) + asal'# C(rama1) +
endogena = 'cotiza'

model = pf.feols(f'{endogena}~{exogenas_formula}', data=df_2022, vcov='hetero')

output = results.joinpath('chile/other_regressions/formalidad_chile.docx')
dis_res.great_table_to_word(pf.etable(model, **table_args), output)
os.startfile(output)

#### REGRESION OCUPACIONES - CHILE

In [ ]:
file = bases.joinpath('chile/limpio/chile_region_processed.sav')
df, var_labels, col_labels = da.read_file(file)
da.var_labels=var_labels
from codigo.chile import filters_chile
import importlib
importlib.reload(filters_chile)
admin, elementary = [1,2,3,4], [9]
filters = (
    c.oficio1_08.is_between(1,9) #0 no porque es de las fuerzas armadas
)
df=df.with_columns(
    pl.when(c.oficio1_08.is_in(admin)).then(1)
    .when(c.oficio1_08.is_in(elementary)).then(0)
    .otherwise(None)
    .alias('recode_ocupacion')
    )

temp_df=df.filter(filters, filters_chile.wage_filters())
fixed_effects='anno+habilidad'
cluster='habilidad'
ocupations = {1:'Admin', 0:'Elementary'}
outputs={}
print(f'DataBase -> {file.name}')
for i in [1,0]:
    print(f'\nRunning regressions \nGroup -> {ocupations.get(i)}')
    reg_df = temp_df.filter(c.recode_ocupacion.eq(i))
    outputs[ocupations.get(i)] = pf.feols(f'w_real_log ~ {filters_chile.exogenas} | {fixed_effects}', data=reg_df, vcov={'CRV1': cluster})
    print(f'\nDone!\n{"*"*20}')

dis_res.great_table_to_word(pf.etable(outputs.values(), model_heads=['Admin', 'Elementary'], **dis_res.table_args), output_path=results.joinpath('chile/regresiones_ocupaciones.docx'))

In [78]:
vars = {2015:'o29', 2017:'o29', 2020:'o32', 2022:'o32'}
for y in [2015,2017,2020,2022]:
    print(y)
    file = bases.joinpath(f'chile/testing/casen_{y}.sav')
    df, var_labels, col_labels = da.read_file(file, metadata_only=True)
    display(var_labels[vars.get(y)])

2015


{1.0: 'Sí, AFP. Cotización obligatoria del trabajador dependiente',
 2.0: 'Sí, AFP. Cotización voluntaria del trabajador independiente',
 3.0: 'Sí, IPS ex INP (CANAEMPU, EMPART, SSS u otras)',
 4.0: 'Sí, Caja de Previsión de la Defensa Nacional (CAPREDENA)',
 5.0: 'Sí, Dirección de Previsión de Carabineros (DIPRECA)',
 6.0: 'Sí, otra. Especifique',
 7.0: 'No está cotizando',
 9.0: 'No sabe'}

2017


{1.0: 'Sí, AFP (Administradora de Fondos de Pensiones). Cotización',
 2.0: 'Sí, AFP (Administradora de Fondos de Pensiones). Cotización',
 3.0: 'Sí, IPS ex INP [Caja Nacional de Empleados Públicos (CANAEMP',
 4.0: 'Sí, Caja de Previsión de la Defensa Nacional (CAPREDENA)',
 5.0: 'Sí, Dirección de Previsión de Carabineros (DIPRECA)',
 6.0: 'Sí, otra. Especifique',
 7.0: 'No está cotizando',
 9.0: 'No sabe'}

2020


{1.0: 'Sí, AFP (Administradora de Fondos de Pensiones)',
 2.0: 'Sí, IPS ex INP [Caja Nacional de Empleados Públicos (CANAEMPU), Caja de Empleados Particulares (EMPART), Servicio de S',
 3.0: 'Sí, Caja de Previsión de la Defensa Nacional (CAPREDENA)',
 4.0: 'Sí, Dirección de Previsión de Carabineros (DIPRECA)',
 5.0: 'Sí, otra. Especifique',
 6.0: 'No está cotizando',
 9.0: 'No sabe'}

2022


{-88.0: 'No sabe',
 1.0: '1. Sí, AFP (Administradora de Fondos de Pensiones)',
 2.0: '2. Sí, IPS ex INP',
 3.0: '3. Sí, Caja de Previsión de la Defensa Nacional (CAPREDENA)',
 4.0: '4. Sí, Dirección de Previsión de Carabineros (DIPRECA)',
 5.0: '5. Sí, otro. Especifique',
 6.0: '6. No está cotizando'}

In [68]:
df, var_labels, col_labels = da.read_file(file)

In [76]:
df.group_by('cotiza o32'.split()).len().sort('cotiza o32'.split())

cotiza,o32,len
f64,f64,u32
null,null,35838
-88.0,null,1185
-88.0,-88.0,481
0.0,null,54474
0.0,6.0,45195
1.0,1.0,61568
1.0,2.0,1924
1.0,3.0,712
1.0,4.0,510
